# Upper PPO Scenario Topology And Variety

This notebook reads an upper PPO `training_log.csv` and plots the fixed 3-gNB topology, snapshot target-load scenarios, sampled scenario variety, and the policy's load/bias behavior.

Use the project virtual environment as the notebook kernel.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "global_ppo_3gnb_env.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

from global_ppo_3gnb_env import (
    DEFAULT_GNB_CONFIGS_3,
    GLOBAL_SNAPSHOT_SCENARIOS,
    GlobalPPO3GNBEnv,
    SLICE_TYPES,
)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
})
np.set_printoptions(precision=3, suppress=True)
print(f"Project root: {PROJECT_ROOT}")

## Load The Training CSV

Set `TRAINING_CSV` manually to inspect another run. If it is `None`, the notebook uses the newest `models/upper_ppo_3gnb/run_*/training_log.csv`.

In [ ]:
TRAINING_CSV = None

if TRAINING_CSV is None:
    candidates = sorted(
        (PROJECT_ROOT / "models" / "upper_ppo_3gnb").glob("run_*/training_log.csv"),
        key=lambda p: p.stat().st_mtime,
    )
    if not candidates:
        raise FileNotFoundError("No upper PPO training_log.csv found under models/upper_ppo_3gnb")
    TRAINING_CSV = candidates[-1]
else:
    TRAINING_CSV = Path(TRAINING_CSV)
    if not TRAINING_CSV.is_absolute():
        TRAINING_CSV = PROJECT_ROOT / TRAINING_CSV

df = pd.read_csv(TRAINING_CSV)
print(TRAINING_CSV)
print(f"rows={len(df):,}, episodes={df['episode'].nunique():,}, scenarios={df['scenario_name'].nunique():,}")
display(df.head(3))

In [ ]:
GNB_IDS = [0, 1, 2]
SLICE_COLORS = {"eMBB": "#2b6cb0", "URLLC": "#c53030", "mMTC": "#2f855a"}
GNB_POS = {
    int(cfg["id"]): (float(cfg["x"]), float(cfg["y"]), float(cfg["coverage_radius"]))
    for cfg in DEFAULT_GNB_CONFIGS_3
}

def matrix_from_columns(row, prefix):
    return np.asarray(
        [[float(row[f"{prefix}_g{g}_{s}"]) for s in SLICE_TYPES] for g in GNB_IDS],
        dtype=float,
    )

def scenario_target(name):
    rows = df[df["scenario_name"] == name]
    if len(rows) and all(f"target_load_g0_{s}" in df.columns for s in SLICE_TYPES):
        return matrix_from_columns(rows.iloc[0], "target_load")
    return np.asarray(GLOBAL_SNAPSHOT_SCENARIOS[name], dtype=float)

def draw_base_topology(ax):
    for g, (x, y, radius) in GNB_POS.items():
        ax.add_patch(Circle((x, y), radius, fill=False, linestyle="--", linewidth=1.2, alpha=0.35))
        ax.scatter([x], [y], s=260, color="#222222", marker="^", zorder=5)
        ax.text(x, y + 28, f"gNB {g}", ha="center", va="bottom", fontweight="bold")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x position")
    ax.set_ylabel("y position")

def draw_load_markers(ax, matrix, title):
    offsets = {"eMBB": (-36, -28), "URLLC": (36, -28), "mMTC": (0, 45)}
    draw_base_topology(ax)
    for g, (x, y, _radius) in GNB_POS.items():
        for s_idx, slice_type in enumerate(SLICE_TYPES):
            value = float(matrix[g, s_idx])
            dx, dy = offsets[slice_type]
            ax.scatter(
                [x + dx], [y + dy],
                s=260 + 1250 * value,
                color=SLICE_COLORS[slice_type],
                alpha=0.72,
                edgecolor="white",
                linewidth=1.2,
                zorder=6,
            )
            ax.text(x + dx, y + dy, f"{value:.2f}", ha="center", va="center", color="white", fontsize=8, fontweight="bold", zorder=7)
    handles = [plt.Line2D([0], [0], marker="o", color="w", label=s, markerfacecolor=SLICE_COLORS[s], markersize=9) for s in SLICE_TYPES]
    ax.legend(handles=handles, loc="upper right")
    ax.set_title(title)

## Topology For One Scenario

Each colored bubble is one slice load at one gNB. Bigger means heavier target load.

In [ ]:
SCENARIO_NAME = df["scenario_name"].value_counts().index[0]

target = scenario_target(SCENARIO_NAME)
fig, ax = plt.subplots(figsize=(8, 6.8))
draw_load_markers(ax, target, f"Topology target loads: {SCENARIO_NAME}")

for s_idx, slice_type in enumerate(SLICE_TYPES):
    overloaded = np.where(target[:, s_idx] >= 0.80)[0]
    light = np.where(target[:, s_idx] <= 0.25)[0]
    for src in overloaded:
        if len(light) == 0:
            continue
        dst = int(light[np.argmin(target[light, s_idx])])
        x0, y0, _ = GNB_POS[int(src)]
        x1, y1, _ = GNB_POS[int(dst)]
        ax.annotate(
            "",
            xy=(x1, y1),
            xytext=(x0, y0),
            arrowprops={"arrowstyle": "->", "lw": 2.2, "color": SLICE_COLORS[slice_type], "alpha": 0.65},
        )

plt.show()

## UE Placement Snapshot

This cell resets the actual training environment on the selected snapshot and plots fixed UE positions. The UEs do not move because snapshot training uses `vx = vy = 0`.

In [ ]:
env = GlobalPPO3GNBEnv(
    seed=7,
    scenario_mode="snapshot",
    snapshot_scenario=SCENARIO_NAME,
    local_steps_per_global=1,
    global_steps_per_episode=3,
)
obs, info = env.reset()

ue_rows = []
for ue in env.base_env.get_all_ues():
    ue_rows.append({
        "ue_id": int(ue.id),
        "x": float(ue.x),
        "y": float(ue.y),
        "slice_type": str(getattr(ue, "slice_type", "eMBB")),
        "serving_gnb": int(ue.serving_gnb) if ue.serving_gnb is not None else -1,
    })
env.close()

ue_df = pd.DataFrame(ue_rows)
display(ue_df.sort_values(["serving_gnb", "slice_type", "ue_id"]).head(20))

fig, ax = plt.subplots(figsize=(8, 6.8))
draw_load_markers(ax, target, f"UE placement: {SCENARIO_NAME}")
for slice_type, rows in ue_df.groupby("slice_type"):
    ax.scatter(rows["x"], rows["y"], s=42, color=SLICE_COLORS.get(slice_type, "gray"), edgecolor="black", linewidth=0.5, label=f"UE {slice_type}", zorder=8)
    for _, row in rows.iterrows():
        ax.text(row["x"] + 8, row["y"] + 8, f"u{int(row['ue_id'])}/g{int(row['serving_gnb'])}", fontsize=7)
plt.show()

## Scenario Variety In This Run

In [ ]:
episode_scenarios = df.groupby("episode")["scenario_name"].first()
counts = episode_scenarios.value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4.8))
counts.plot(kind="barh", ax=ax, color="#4a5568")
ax.set_xlabel("episodes")
ax.set_ylabel("scenario")
ax.set_title("How often each snapshot scenario appears")
plt.show()

display(counts.sort_values(ascending=False).rename("episodes").to_frame())

In [ ]:
scenario_names = list(counts.sort_values(ascending=False).index)
n = len(scenario_names)
cols = 3
rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.2, rows * 3.6), squeeze=False)

for ax, name in zip(axes.flat, scenario_names):
    matrix = scenario_target(name)
    im = ax.imshow(matrix, vmin=0.0, vmax=1.0, cmap="viridis")
    ax.set_title(name)
    ax.set_xticks(range(len(SLICE_TYPES)), SLICE_TYPES)
    ax.set_yticks(range(len(GNB_IDS)), [f"g{g}" for g in GNB_IDS])
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", color="white", fontsize=8, fontweight="bold")

for ax in axes.flat[n:]:
    ax.axis("off")

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.75, label="target load")
fig.suptitle("Target-load matrix for every scenario", y=1.02, fontsize=14)
plt.show()

## End-Of-Episode Behavior

Because reward is terminal-only, the most important rows are `done == True`. These plots summarize what happened at the end of each episode.

In [ ]:
done_mask = df["done"].astype(str).str.lower().isin(["true", "1", "yes"])
terminal = df[done_mask].copy()
if terminal.empty:
    terminal = df.groupby("episode", as_index=False).tail(1).copy()

summary_cols = ["reward", "load_variance", "overload_ratio", "sla_count", "handover_count"]
summary = terminal.groupby("scenario_name")[summary_cols].mean().sort_values("reward", ascending=False)
display(summary.round(3))

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
summary["reward"].sort_values().plot(kind="barh", ax=axes[0], color="#2b6cb0", title="terminal reward")
summary["load_variance"].sort_values().plot(kind="barh", ax=axes[1], color="#805ad5", title="final load variance")
summary["handover_count"].sort_values().plot(kind="barh", ax=axes[2], color="#dd6b20", title="final handovers")
for ax in axes:
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
def scenario_mean_matrix(frame, prefix):
    return np.asarray(
        [[frame[f"{prefix}_g{g}_{s}"].mean() for s in SLICE_TYPES] for g in GNB_IDS],
        dtype=float,
    )

SCENARIO_TO_COMPARE = summary.index[0] if len(summary) else SCENARIO_NAME
scenario_terminal = terminal[terminal["scenario_name"] == SCENARIO_TO_COMPARE]

matrices = {
    "target load": scenario_mean_matrix(scenario_terminal, "target_load"),
    "final load": scenario_mean_matrix(scenario_terminal, "load"),
    "final bias": scenario_mean_matrix(scenario_terminal, "bias"),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (title, matrix) in zip(axes, matrices.items()):
    cmap = "coolwarm" if "bias" in title else "viridis"
    vmin, vmax = (-1, 1) if "bias" in title else (0, 1)
    im = ax.imshow(matrix, vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_title(title)
    ax.set_xticks(range(len(SLICE_TYPES)), SLICE_TYPES)
    ax.set_yticks(range(len(GNB_IDS)), [f"g{g}" for g in GNB_IDS])
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", color="white", fontsize=8, fontweight="bold")
    fig.colorbar(im, ax=ax, shrink=0.75)
fig.suptitle(f"Mean terminal matrices: {SCENARIO_TO_COMPARE}", y=1.05, fontsize=14)
plt.show()

## One Episode Timeline

In [ ]:
EPISODE = int(df["episode"].iloc[0])
episode_df = df[df["episode"] == EPISODE].copy()

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
axes[0].plot(episode_df["episode_step"], episode_df["load_variance"], marker="o")
axes[0].set_ylabel("load variance")
axes[1].plot(episode_df["episode_step"], episode_df["overload_ratio"], marker="o", color="#c53030")
axes[1].set_ylabel("overload ratio")
axes[2].plot(episode_df["episode_step"], episode_df["handover_count"], marker="o", color="#2f855a")
axes[2].set_ylabel("handovers")
axes[2].set_xlabel("upper step inside episode")
fig.suptitle(f"Episode {EPISODE}: {episode_df['scenario_name'].iloc[0]}")
plt.tight_layout()
plt.show()